# Unit 1 — REINFORCE → Vanilla Policy Gradient

> *Part of the [RL for Robotics & LLMs](https://github.com/AliBuildsAI/rl-for-robotics-llms) series.*

We start with **REINFORCE** (Williams, 1992) — the original policy gradient
algorithm — then build up to **Vanilla Policy Gradient (VPG)** by adding a
learned value-function baseline, and show that the baseline directly improves
training stability.

This notebook follows **Algorithm 1** from Schulman et al. exactly.


In [ ]:
%pip install -q "gymnasium[classic-control]" imageio pillow matplotlib torch
print("All packages ready.")


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import imageio.v3 as iio
from IPython.display import Image, display

print(f"gymnasium {gym.__version__}  |  torch {torch.__version__}")


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


---
## 1  The Environment — CartPole-v1

CartPole is the "Hello World" of RL. A pole is attached to a cart on a
frictionless track. The agent must keep the pole upright by pushing left or right.

**State** (4 floats):

| Index | Variable | Meaning |
|-------|----------|---------|
| 0 | `x` | Cart position |
| 1 | `ẋ` | Cart velocity |
| 2 | `θ` | Pole angle (radians) |
| 3 | `θ̇` | Pole angular velocity |

**Actions**: 0 = push left, 1 = push right
**Reward**: +1 every timestep the pole stays up
**Solved**: greedy policy averages ≥ 475 reward over 100 episodes


---
## 2  Algorithm 1 — Vanilla Policy Gradient

The algorithm we implement in this notebook, verbatim:

```
Initialize policy parameter θ, baseline b
for iteration = 1, 2, … do
    Collect trajectories by executing the current policy
    At each timestep, compute:
        return          R_t = Σ_{t'=t}^{T-1} γ^{t'-t} r_{t'}
        advantage est.  Â_t = R_t − b(s_t)
    Re-fit the baseline by minimising ‖b(s_t) − R_t‖²
    Update policy using gradient estimate ĝ = Σ_t ∇_θ log π(a_t|s_t, θ) Â_t
end for
```

We build this up in two stages:

| Stage | Baseline | Advantage |
|-------|----------|-----------|
| **Part A — REINFORCE** | none (b = 0) | Â_t = R_t |
| **Part B — VPG** | learned V(s) | Â_t = R_t − V(s_t) |

### Why does a baseline help?

The policy gradient is unbiased for *any* baseline b(s) that doesn't depend on
the action.  But it dramatically **reduces variance**.

Intuitively: instead of asking *"was this episode good?"* (which depends on
luck), we ask *"was this episode better than expected from this state?"*.
The value function $V(s_t)$ provides that expectation.


---
## 3  Policy Network


In [ ]:
class PolicyNetwork(nn.Module):
    """
    Maps state → action probability distribution.

    Input:  state  (state_dim,)     e.g. (4,) for CartPole
    Output: probs  (action_dim,)    sums to 1
    """

    def __init__(self, state_dim: int, action_dim: int, hidden_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),   # (4,)   → (128,)
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),  # (128,) → (128,)
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim),  # (128,) → (2,)  raw logits
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x:       (..., state_dim)
        # returns: (..., action_dim)  — valid probability distribution
        return torch.softmax(self.net(x), dim=-1)

    def select_action(self, state: np.ndarray):
        """
        Sample one action; return (action, log_prob).

        log_prob is a scalar Tensor with grad_fn — PyTorch will backprop
        through it when we call loss.backward().
        """
        state_t = torch.as_tensor(state, dtype=torch.float32, device=device)
        # state_t: (state_dim,)

        probs  = self.forward(state_t)           # (action_dim,)
        dist   = torch.distributions.Categorical(probs)
        action = dist.sample()                   # scalar Tensor

        return action.item(), dist.log_prob(action)


---
## 4  Reward-to-Go

$$R_t = \sum_{t'=t}^{T-1} \gamma^{t'-t}\, r_{t'}$$

This is the "reward-to-go" — the discounted sum of *future* rewards from step
$t$ onward.  Actions at step $t$ can't affect the past, so we discard those
earlier rewards. This reduces variance without introducing bias.


In [ ]:
def compute_returns(rewards: list, gamma: float = 0.99) -> torch.Tensor:
    """
    Compute discounted reward-to-go for one episode.

    Returns a Tensor of shape (T,) on `device`.  Values are NOT normalised —
    the algorithm uses raw R_t.  Baseline subtraction handles variance reduction.

    Example (gamma=1, rewards=[1,1,1]):
        G_2 = 1
        G_1 = 1 + 1·G_2 = 2
        G_0 = 1 + 1·G_1 = 3   →  tensor([3., 2., 1.])
    """
    G, returns = 0.0, []
    for r in reversed(rewards):      # walk backwards through the episode
        G = r + gamma * G
        returns.insert(0, G)

    return torch.tensor(returns, dtype=torch.float32, device=device)
    # shape: (T,)


---
## Part A — REINFORCE (baseline b = 0)

With no baseline, the advantage estimate is just the raw return:

$$\hat{A}_t = R_t$$

The policy loss is:

$$\mathcal{L} = -\sum_t \log \pi_\theta(a_t \mid s_t) \cdot R_t$$

This is unbiased but high-variance — returns depend on the full episode length,
which is noisy early in training.


In [ ]:
def reinforce(
    env:          gym.Env,
    policy:       PolicyNetwork,
    optimizer:    torch.optim.Optimizer,
    n_episodes:   int   = 1500,
    gamma:        float = 0.99,
    print_every:  int   = 100,
) -> list:
    """
    REINFORCE — Algorithm 1 with b = 0 (no baseline).

    Exactly the policy gradient update:
        g = Σ_t ∇_θ log π(a_t|s_t) · R_t
    """
    episode_rewards = []

    for ep in range(1, n_episodes + 1):

        # ── Collect one trajectory ────────────────────────────────────────────
        state, _ = env.reset()
        log_probs: list[torch.Tensor] = []
        rewards:   list[float]        = []
        done = False

        while not done:
            action, log_prob = policy.select_action(state)
            state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            log_probs.append(log_prob)
            rewards.append(reward)

        # ── Compute returns R_t ───────────────────────────────────────────────
        returns = compute_returns(rewards, gamma)
        # returns: (T,)  raw discounted reward-to-go, no normalisation

        # ── Policy gradient loss ──────────────────────────────────────────────
        # Â_t = R_t  (baseline b = 0)
        # L   = -Σ_t log π(a_t|s_t) · R_t
        loss = -torch.stack(
            [lp * R for lp, R in zip(log_probs, returns)]
        ).sum()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        episode_rewards.append(sum(rewards))

        if ep % print_every == 0:
            avg = np.mean(episode_rewards[-print_every:])
            print(f"[REINFORCE]  ep {ep:5d}/{n_episodes}  avg: {avg:6.1f}")

    return episode_rewards


---
## 5  Train Part A (REINFORCE)

We fix a random seed so the Part A and Part B comparisons are fair — same
environment randomness, same policy initialisation.


In [ ]:
SEED       = 42
N_EPISODES = 1500
GAMMA      = 0.99
LR         = 1e-3
HIDDEN_DIM = 128

def make_env_and_policy(seed):
    """Reproducible environment + freshly-initialised policy."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    env = gym.make("CartPole-v1")
    env.reset(seed=seed)
    state_dim  = env.observation_space.shape[0]   # 4
    action_dim = env.action_space.n               # 2
    policy = PolicyNetwork(state_dim, action_dim, HIDDEN_DIM).to(device)
    return env, policy

env_a, policy_a = make_env_and_policy(SEED)
optimizer_a     = optim.Adam(policy_a.parameters(), lr=LR)

print("=" * 60)
print(f"Part A — REINFORCE (no baseline)  |  {N_EPISODES} episodes")
print("=" * 60)

rewards_reinforce = reinforce(env_a, policy_a, optimizer_a,
                               n_episodes=N_EPISODES, gamma=GAMMA)
env_a.close()
print(f"\nTraining avg (last 100 eps): {np.mean(rewards_reinforce[-100:]):.1f}")


---
## Part B — VPG with Learned Value Baseline

Now we implement the **full** Algorithm 1.  We add a **value network** $V_\phi(s)$
that learns to predict the expected return from each state.

### Two networks, two losses

**Policy network $\pi_\theta$** — same as before, updated with the advantage:
$$\mathcal{L}_{\text{policy}} = -\sum_t \log \pi_\theta(a_t \mid s_t) \cdot \hat{A}_t$$

**Value network $V_\phi$** — outputs a scalar, updated to predict the return:
$$\mathcal{L}_{\text{value}} = \sum_t \bigl(V_\phi(s_t) - R_t\bigr)^2$$

### The advantage
$$\hat{A}_t = R_t - V_\phi(s_t)$$

If the agent is in a good state and acts well, $R_t > V_\phi(s_t)$ → positive
advantage → that action gets reinforced.  If the return is *worse* than
expected, the advantage is negative and that action is suppressed.

This is **much** lower variance than raw $R_t$ because we're measuring
*relative* performance rather than absolute return.


In [ ]:
class ValueNetwork(nn.Module):
    """
    Maps state → scalar value estimate V(s).

    Predicts the expected discounted return from state s.
    Used as the baseline b(s) in Algorithm 1.

    Input:  state (state_dim,)
    Output: scalar ()
    """

    def __init__(self, state_dim: int, hidden_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),   # (4,)   → (128,)
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),  # (128,) → (128,)
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),           # (128,) → (1,)  no activation
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x:       (..., state_dim)
        # returns: (...,)           — squeeze the trailing 1
        return self.net(x).squeeze(-1)


In [ ]:
def vpg_with_baseline(
    env:        gym.Env,
    policy:     PolicyNetwork,
    value_net:  ValueNetwork,
    policy_opt: torch.optim.Optimizer,
    value_opt:  torch.optim.Optimizer,
    n_episodes: int   = 1500,
    gamma:      float = 0.99,
    print_every: int  = 100,
) -> list:
    """
    VPG — Algorithm 1 with a learned value-function baseline.

    For each episode:
      1. Roll out trajectory
      2. Compute returns R_t
      3. Compute advantages Â_t = R_t - V(s_t)
      4. Re-fit value network:  minimise Σ (V(s_t) - R_t)²
      5. Update policy:         minimise -Σ log π(a_t|s_t) · Â_t
    """
    episode_rewards = []

    for ep in range(1, n_episodes + 1):

        # ── 1. Collect one trajectory ─────────────────────────────────────────
        state, _ = env.reset()
        states:    list[np.ndarray]   = []
        log_probs: list[torch.Tensor] = []
        rewards:   list[float]        = []
        done = False

        while not done:
            action, log_prob = policy.select_action(state)
            states.append(state)                 # save s_t to compute V(s_t)
            log_probs.append(log_prob)
            state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            rewards.append(reward)

        # ── 2. Compute returns R_t ────────────────────────────────────────────
        returns = compute_returns(rewards, gamma)
        # returns: (T,)

        # ── 3. Compute advantages  Â_t = R_t - V(s_t) ────────────────────────
        states_t = torch.as_tensor(
            np.array(states), dtype=torch.float32, device=device
        )
        # states_t: (T, state_dim)

        values = value_net(states_t)
        # values: (T,)  — V_φ(s_t) for each timestep

        # detach values so the policy gradient doesn't flow into the value net
        advantages = returns - values.detach()
        # advantages: (T,)  — Â_t

        # ── 4. Re-fit value network ───────────────────────────────────────────
        # Minimise  ‖V(s_t) - R_t‖²  (Algorithm 1, step 4)
        value_loss = F.mse_loss(values, returns.detach())

        value_opt.zero_grad()
        value_loss.backward()
        value_opt.step()

        # ── 5. Update policy ──────────────────────────────────────────────────
        # g = Σ_t ∇_θ log π(a_t|s_t) · Â_t
        policy_loss = -torch.stack(
            [lp * A for lp, A in zip(log_probs, advantages)]
        ).sum()

        policy_opt.zero_grad()
        policy_loss.backward()
        policy_opt.step()

        episode_rewards.append(sum(rewards))

        if ep % print_every == 0:
            avg = np.mean(episode_rewards[-print_every:])
            print(f"[VPG+baseline]  ep {ep:5d}/{n_episodes}  avg: {avg:6.1f}")

    return episode_rewards


---
## 6  Train Part B (VPG with Baseline)

Same seed, same architecture, same number of episodes — only difference is the
value network baseline.


In [ ]:
env_b, policy_b = make_env_and_policy(SEED)
value_net   = ValueNetwork(env_b.observation_space.shape[0], HIDDEN_DIM).to(device)
policy_opt  = optim.Adam(policy_b.parameters(), lr=LR)
value_opt   = optim.Adam(value_net.parameters(), lr=LR)

print("=" * 60)
print(f"Part B — VPG with value baseline  |  {N_EPISODES} episodes")
print("=" * 60)

rewards_vpg = vpg_with_baseline(
    env_b, policy_b, value_net, policy_opt, value_opt,
    n_episodes=N_EPISODES, gamma=GAMMA,
)
env_b.close()
print(f"\nTraining avg (last 100 eps): {np.mean(rewards_vpg[-100:]):.1f}")


---
## 7  Comparison — REINFORCE vs VPG with Baseline

Both curves are smoothed with a 50-episode window.
The dashed red line is the "solved" threshold (475).


In [ ]:
def moving_average(values: list, window: int = 50) -> list:
    out = []
    for i in range(len(values)):
        lo = max(0, i - window + 1)
        out.append(float(np.mean(values[lo : i + 1])))
    return out


fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
eps = range(1, N_EPISODES + 1)

for ax, rewards, label, color in [
    (axes[0], rewards_reinforce, "REINFORCE (b = 0)",        "steelblue"),
    (axes[1], rewards_vpg,       "VPG + value baseline",     "darkorange"),
]:
    ax.plot(eps, rewards, alpha=0.2, color=color, linewidth=0.8)
    ax.plot(eps, moving_average(rewards, 50), color=color,
            linewidth=2.5, label=label)
    ax.axhline(475, color="red", linestyle="--", alpha=0.7,
               label="Solved (475)")
    ax.set_xlabel("Episode")
    ax.set_ylabel("Total Reward")
    ax.set_title(label, fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.25)

plt.suptitle("REINFORCE vs VPG with Value Baseline", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("comparison.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved → comparison.png")


In [ ]:
def evaluate_greedy(policy: PolicyNetwork, n_episodes: int = 100) -> float:
    """Run n_episodes greedy (argmax) and return mean total reward."""
    eval_env = gym.make("CartPole-v1")
    rewards  = []
    policy.eval()
    with torch.no_grad():
        for _ in range(n_episodes):
            state, _ = eval_env.reset()
            total, done = 0.0, False
            while not done:
                state_t = torch.as_tensor(state, dtype=torch.float32, device=device)
                action  = policy(state_t).argmax().item()
                state, r, terminated, truncated, _ = eval_env.step(action)
                done = terminated or truncated
                total += r
            rewards.append(total)
    eval_env.close()
    policy.train()
    return float(np.mean(rewards))


score_a = evaluate_greedy(policy_a)
score_b = evaluate_greedy(policy_b)

print("Greedy evaluation (100 episodes, argmax policy):")
print(f"  REINFORCE (b = 0)     : {score_a:.1f}  {'✓ SOLVED' if score_a >= 475 else 'not solved'}")
print(f"  VPG + value baseline  : {score_b:.1f}  {'✓ SOLVED' if score_b >= 475 else 'not solved'}")


---
## 8  Watch the VPG Agent


In [ ]:
def record_gif(policy: PolicyNetwork, filename: str = "agent.gif",
               max_steps: int = 500) -> str:
    """Greedy episode → animated GIF. render_mode='rgb_array' works headless."""
    env_rec = gym.make("CartPole-v1", render_mode="rgb_array")
    frames, total_reward = [], 0.0

    state, _ = env_rec.reset()
    done = False

    policy.eval()
    with torch.no_grad():
        while not done and len(frames) < max_steps:
            frames.append(env_rec.render())            # (H, W, 3) uint8
            state_t = torch.as_tensor(state, dtype=torch.float32, device=device)
            action  = policy(state_t).argmax().item()  # greedy
            state, reward, terminated, truncated, _ = env_rec.step(action)
            done = terminated or truncated
            total_reward += reward

    env_rec.close()
    policy.train()

    frames_arr = np.stack(frames)                      # (N, H, W, 3)
    iio.imwrite(filename, frames_arr, plugin="pillow", loop=0, duration=33)
    print(f"{len(frames)} frames  |  total reward: {total_reward:.0f}  →  {filename}")
    return filename


gif_path = record_gif(policy_b)   # record the VPG agent
display(Image(gif_path))


---
## 9  What's Next

| Limitation | Fix | Unit |
|-----------|-----|------|
| Monte Carlo returns are high variance | **GAE** — exponentially-weighted mix of 1-step and MC returns | Unit 2 |
| One gradient step per episode | Importance sampling → **PPO** reuses each batch for multiple epochs | Unit 3 |
| Reward comes from environment | Learn reward from human preferences → **RLHF** | Unit 4 |
| Use RL to teach *reasoning* | **GRPO** — reward model judges chain-of-thought correctness | Unit 5 |

The key insight carried forward: we're always optimising
$\mathbb{E}[\log \pi_\theta(a|s) \cdot \hat{A}]$
and the series is about building better and better estimates of $\hat{A}$.

→ [Back to the series](https://github.com/AliBuildsAI/rl-for-robotics-llms)
